<a href="https://colab.research.google.com/github/vnandini867-ui/vnandini867/blob/main/LANGGRAPH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q -U langchain-groq langgraph


# ============================================================
# IMPORTS
# ============================================================

import os
from typing import TypedDict, Literal

from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

from langgraph.graph import StateGraph, END, START


# ============================================================
# LOAD GROQ API KEY FROM GOOGLE COLAB SECRETS
# ============================================================

try:
    GROQ_API_KEY = userdata.get("nandu")

    if not GROQ_API_KEY:
        raise ValueError("API key is empty")

    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

    print("Groq API key loaded successfully.")

except Exception as e:
    print("ERROR: Groq API key 'nandu' was not found.")
    print("Go to Colab → Secrets → create a secret named 'nandu'.")
    raise e


# ============================================================
# INITIALIZE GROQ LLM
# ============================================================

# ============================================================
# INITIALIZE GROQ LLM
# ============================================================

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq LLM initialized successfully.")


# ============================================================
# 1. STATE DEFINITION
# ============================================================

class AgentState(TypedDict):
    user_query: str
    category: str
    final_answer: str


# ============================================================
# 2. CLASSIFICATION NODE
# ============================================================

def classify_node(state: AgentState) -> dict:

    prompt = (
        "Classify the following query into exactly one word:\n"
        "'code' if it asks for programming, coding, debugging, "
        "software development, Python, Java, C, C++, HTML, CSS, "
        "JavaScript, SQL, algorithms, or another programming topic.\n"
        "'general' otherwise.\n\n"
        f"Query: {state['user_query']}"
    )

    response = llm.invoke([
        HumanMessage(content=prompt)
    ])

    result = response.content.strip().lower()

    if "code" in result:
        category = "code"
    else:
        category = "general"

    return {
        "category": category
    }


# ============================================================
# 3. CODE NODE
# ============================================================

def code_node(state: AgentState) -> dict:

    response = llm.invoke([
        SystemMessage(
            content=(
                "You are an expert software engineer. "
                "Answer programming questions clearly and simply. "
                "Provide correct code when required. "
                "Explain important parts of the code."
            )
        ),
        HumanMessage(
            content=state["user_query"]
        )
    ])

    return {
        "final_answer": response.content
    }


# ============================================================
# 4. GENERAL NODE
# ============================================================

def general_node(state: AgentState) -> dict:

    response = llm.invoke([
        SystemMessage(
            content=(
                "You are a helpful general assistant. "
                "Give clear, accurate and easy-to-understand answers."
            )
        ),
        HumanMessage(
            content=state["user_query"]
        )
    ])

    return {
        "final_answer": response.content
    }


# ============================================================
# 5. ROUTING FUNCTION
# ============================================================

def route_by_category(
    state: AgentState
) -> Literal["code_node", "general_node"]:

    if state["category"] == "code":
        return "code_node"

    return "general_node"


# ============================================================
# 6. CREATE LANGGRAPH
# ============================================================

builder = StateGraph(AgentState)

builder.add_node(
    "classify",
    classify_node
)

builder.add_node(
    "code_node",
    code_node
)

builder.add_node(
    "general_node",
    general_node
)


# ============================================================
# 7. CONNECT GRAPH
# ============================================================

builder.add_edge(
    START,
    "classify"
)

builder.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "code_node": "code_node",
        "general_node": "general_node"
    }
)

builder.add_edge(
    "code_node",
    END
)

builder.add_edge(
    "general_node",
    END
)


# ============================================================
# 8. COMPILE GRAPH
# ============================================================

graph = builder.compile()

print("LangGraph compiled successfully.")


# ============================================================
# 9. TEST 1 — PROGRAMMING QUESTION
# ============================================================

query_1 = "How do I reverse a list in Python?"

print("\n" + "=" * 60)
print("QUERY 1")
print("=" * 60)

print("Question:", query_1)

result_1 = graph.invoke({
    "user_query": query_1,
    "category": "",
    "final_answer": ""
})

print("\nRouted Category:")
print(result_1["category"])

print("\nResponse:")
print(result_1["final_answer"])


# ============================================================
# 10. TEST 2 — GENERAL QUESTION
# ============================================================

query_2 = "What causes rain to fall?"

print("\n" + "=" * 60)
print("QUERY 2")
print("=" * 60)

print("Question:", query_2)

result_2 = graph.invoke({
    "user_query": query_2,
    "category": "",
    "final_answer": ""
})

print("\nRouted Category:")
print(result_2["category"])

print("\nResponse:")
print(result_2["final_answer"])

Groq API key loaded successfully.
Groq LLM initialized successfully.
LangGraph compiled successfully.

QUERY 1
Question: How do I reverse a list in Python?

Routed Category:
code

Response:
You can reverse a list in Python in several ways, depending on whether you want to **modify the original list in place** or **create a new reversed copy**.

---

## 1. In‑place reversal (`list.reverse()`)

```python
numbers = [1, 2, 3, 4, 5]

# This changes `numbers` itself
numbers.reverse()

print(numbers)          # → [5, 4, 3, 2, 1]
```

* **What it does:** Calls the list’s built‑in `reverse` method, which rearranges the elements of the list in reverse order.
* **Side effect:** The original list is mutated; no new list is created.
* **Complexity:** O(n) time, O(1) extra space.

---

## 2. Creating a new reversed list with slicing

```python
numbers = [1, 2, 3, 4, 5]

reversed_copy = numbers[::-1]   # slice step -1

print(reversed_copy)   # → [5, 4, 3, 2, 1]
print(numbers)         # original uncha